In [1]:
import os
import json
import cv2
import numpy as np
import shutil
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/GIST514"
target_path = r"/mnt/lustre/lalit47/down/seg/GIST"

# Source folders/files
images_folder = os.path.join(source_path, "images")
annotations_folder = os.path.join(source_path, "annotations")

train_json = os.path.join(annotations_folder, "train_anno_crop_split_0.json")
val_json = os.path.join(annotations_folder, "val_anno_crop_split_0.json")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")
test_masks_out = os.path.join(target_path, "test", "masks")

# Create target directories
for folder in [
    train_images_out,
    train_masks_out,
    val_images_out,
    val_masks_out,
    test_images_out,
    test_masks_out,
]:
    os.makedirs(folder, exist_ok=True)


def process_and_save_image(src, dst):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read image: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)

    return True


def resize_and_save_mask(mask, dst):
    """
    Resize mask to 256x256 using nearest-neighbor
    and save as PNG.
    """

    mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(dst, mask)

    return True


def create_masks_from_coco(json_file, image_output, mask_output):
    """
    Create image and mask pairs from COCO-style JSON annotations.

    Expected JSON format:
    {
        "images": [
            {
                "id": 155,
                "file_name": "original_gist155_1.jpg",
                "height": 446,
                "width": 531
            }
        ],
        "annotations": [
            {
                "image_id": 155,
                "segmentation": [[x1, y1, x2, y2, ...]]
            }
        ]
    }
    """

    with open(json_file, "r") as f:
        data = json.load(f)

    images = data["images"]
    annotations = data["annotations"]

    # Map image_id to image info
    image_id_to_info = {}
    for img in images:
        image_id_to_info[img["id"]] = img

    # Group annotations by image_id
    ann_by_image_id = defaultdict(list)
    for ann in annotations:
        ann_by_image_id[ann["image_id"]].append(ann)

    print(f"\nProcessing: {json_file}")
    print(f"Total images in JSON: {len(images)}")
    print(f"Total annotations in JSON: {len(annotations)}")

    processed_count = 0
    missing_count = 0

    for img in images:
        image_id = img["id"]
        file_name = img["file_name"]
        width = img["width"]
        height = img["height"]

        image_src = os.path.join(images_folder, file_name)

        stem = os.path.splitext(file_name)[0]
        image_dst = os.path.join(image_output, stem + ".png")
        mask_dst = os.path.join(mask_output, stem + ".png")

        # Save image
        if os.path.exists(image_src):
            process_and_save_image(image_src, image_dst)
        else:
            print(f"Image Missing: {image_src}")
            missing_count += 1
            continue

        # Create empty mask
        mask = np.zeros((height, width), dtype=np.uint8)

        # Draw all polygons for this image
        image_annotations = ann_by_image_id.get(image_id, [])

        for ann in image_annotations:
            segmentation = ann.get("segmentation", [])

            # COCO polygon format: segmentation is usually list of polygons
            for polygon in segmentation:
                if len(polygon) < 6:
                    continue

                points = np.array(polygon, dtype=np.float32)
                points = points.reshape(-1, 2)
                points = np.round(points).astype(np.int32)

                cv2.fillPoly(mask, [points], 255)

        # Save mask
        resize_and_save_mask(mask, mask_dst)

        processed_count += 1

    print(f"Processed images: {processed_count}")
    print(f"Missing images: {missing_count}")


def copy_folder(src_folder, dst_folder):
    """
    Copy already processed PNG files from one folder to another.
    Used to make test same as val.
    """

    os.makedirs(dst_folder, exist_ok=True)

    for file in os.listdir(src_folder):
        if file.lower().endswith(".png"):
            src = os.path.join(src_folder, file)
            dst = os.path.join(dst_folder, file)
            shutil.copy2(src, dst)


# =========================
# Main processing
# =========================

# Train from train annotation JSON
create_masks_from_coco(
    train_json,
    train_images_out,
    train_masks_out
)

# Val from val annotation JSON
create_masks_from_coco(
    val_json,
    val_images_out,
    val_masks_out
)

# Test = same as val
copy_folder(val_images_out, test_images_out)
copy_folder(val_masks_out, test_masks_out)

print("\nDataset preprocessing completed successfully!")
print("Train created from train_anno_crop_split_0.json")
print("Val created from val_anno_crop_split_0.json")
print("Test is same as Val")


Processing: /mnt/lustre/lalit47/us_data/bench/GIST514/annotations/train_anno_crop_split_0.json
Total images in JSON: 408
Total annotations in JSON: 408
Processed images: 408
Missing images: 0

Processing: /mnt/lustre/lalit47/us_data/bench/GIST514/annotations/val_anno_crop_split_0.json
Total images in JSON: 106
Total annotations in JSON: 106
Processed images: 106
Missing images: 0

Dataset preprocessing completed successfully!
Train created from train_anno_crop_split_0.json
Val created from val_anno_crop_split_0.json
Test is same as Val
